# 5. Inverse PINNs, Identifiability, and Held-Out Validation

**Contributor:** Rohan Agrawal (WP05)

In this section, we estimate unknown parameters of strong-Allee harvesting models from discrete population observations.
We investigate two central questions:
1. **Structural Identifiability:** Are model parameters uniquely determined by ideal observation trajectories?
2. **Inference & Generalization:** How accurately can parameters and held-out future states be predicted under measurement noise using conventional ODE least squares versus Physics-Informed Neural Networks (PINNs)?

In [ ]:
import csv
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Automatic Colab & Local path detection
if "google.colab" in sys.modules:
    for candidate in [
        Path.cwd(),
        Path("/content"),
        Path("/content/Rohan_RK45_Data_and_Code"),
        Path("/content/mm-allee-harvesting"),
    ]:
        if (candidate / "src/mm_allee").exists():
            if str(candidate / "src") not in sys.path:
                sys.path.insert(0, str(candidate / "src"))
            os.chdir(candidate)
            break
else:
    ROOT = next(
        (
            p
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "src/mm_allee").exists()
        ),
        None,
    )
    if ROOT is not None and str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))

from mm_allee.data import generate_noisy_observations
from mm_allee.inference import fit_ode_least_squares
from mm_allee.models import Parameters
from mm_allee.plotting import save_figure
from mm_allee.solver import solve_trajectory

print("Loaded shared modules successfully.")


wp05_demo_dir = ROOT / "results/wp05_rohan_demo"
wp05_demo_dir.mkdir(parents=True, exist_ok=True)

## 5.1 Model B Structural Nonidentifiability and Counterexample

Under Model B, harvesting is proportional to population size ($h N$):
$$\frac{dN}{dt} = r N \left(1 - \frac{N}{K}\right)\left(\frac{N}{A} - 1\right) - h N$$

Expanding the cubic polynomial:
$$\frac{dN}{dt} = -(r + h) N + r\left(\frac{1}{A} + \frac{1}{K}\right) N^2 - \frac{r}{AK} N^3$$

Any trajectory observed at a single constant effort $h$ reveals at most **three** composite polynomial combinations:
$$c_1 = -(r + h), \quad c_2 = r\left(\frac{1}{A} + \frac{1}{K}\right) = \frac{r(A+K)}{AK}, \quad c_3 = -\frac{r}{AK}$$

Because there are 4 unknown physical parameters $(r, A, K, h)$ but only 3 observable coefficients, the system is **structurally nonidentifiable** from data at a single fixed effort.

In [ ]:
%matplotlib inline

# Demonstrate the exact two-parameter-set counterexample:
p1 = Parameters(r=1.0, A=0.2, K=1.0, h=0.5)
p2 = Parameters(r=1.35, A=0.3, K=0.9, h=0.15)

c1_1, c2_1, c3_1 = -(p1.r + p1.h), p1.r * (1.0 / p1.A + 1.0 / p1.K), -p1.r / (p1.A * p1.K)
c1_2, c2_2, c3_2 = -(p2.r + p2.h), p2.r * (1.0 / p2.A + 1.0 / p2.K), -p2.r / (p2.A * p2.K)

print(f"Parameter Set 1: c1={c1_1:.2f}, c2={c2_1:.2f}, c3={c3_1:.2f}")
print(f"Parameter Set 2: c1={c1_2:.2f}, c2={c2_2:.2f}, c3={c3_2:.2f}")
assert np.isclose(c1_1, c1_2) and np.isclose(c2_1, c2_2) and np.isclose(c3_1, c3_2)

# Solve trajectories from three distinct initial states for both parameter sets
d1, d2 = p1.dimensionless(), p2.dimensionless()
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

for n0, color in zip([0.4, 0.7, 1.1], colors, strict=True):
    traj1 = solve_trajectory(
        "B", a=d1["a"], harvest=d1["e"], x0=n0 / p1.K, tau_end=p1.r * 4.0, n_points=81
    )
    traj2 = solve_trajectory(
        "B", a=d2["a"], harvest=d2["e"], x0=n0 / p2.K, tau_end=p2.r * 4.0, n_points=81
    )
    t1, y1 = traj1.tau / p1.r, traj1.x * p1.K
    t2, y2 = traj2.tau / p2.r, traj2.x * p2.K
    ax.plot(t1, y1, label=f"Set 1 (N0={n0})", color=color, lw=2)
    ax.plot(t2, y2, "--", label=f"Set 2 (N0={n0})", color="black", alpha=0.7)

ax.set_title(
    "Model B Identifiability Counterexample: Identical Trajectories from Different Parameters"
)
ax.set_xlabel("Time t")
ax.set_ylabel("Population N(t)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 5.2 Observations, Noise Generation, and Frozen 60/20/20 Splits

We construct synthetic observations as follows:
$$N_{\text{obs}}(t_i) = N_{\text{clean}}(t_i) + \sigma \cdot N_{\text{ref}} \cdot \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, 1)$$
with fixed external scale $N_{\text{ref}} = 100.0$. Observations are never silently clipped. Each trajectory has **101 observations** on $t\in[0,8]$.

Splits are strictly partitioned into contiguous time windows:
- **Train (60%, 61 points, $t\le4.8$):** used exclusively for fitting and data loss.
- **Validation (20%, 20 points, $4.88\le t\le6.4$):** never in the gradient. The clean demo and frozen benchmark disable checkpoint selection; the noisy live demo explicitly enables validation-based stopping/checkpoint selection.
- **Held-out Test (20%, 20 points, $6.48\le t\le8$):** completely frozen until final post-training evaluation.

In [ ]:
p_true = Parameters(r=0.7, K=100.0, A=20.0, H=10.0)
d_true = p_true.dimensionless()
ref_traj = solve_trajectory(
    "A", a=d_true["a"], harvest=d_true["q"], x0=45.0 / 100.0, tau_end=0.7 * 8.0, n_points=101
)
times = ref_traj.tau / 0.7
clean_pops = ref_traj.x * 100.0

# Generate clean data (sigma = 0.0)
obs_clean = generate_noisy_observations(
    times,
    clean_pops,
    trajectory_id="a_regular_clean",
    model="A",
    noise_sigma=0.0,
    noise_seed=11,
    n_ref=100.0,
)

# Generate 1% noisy data (sigma = 0.01)
obs_1pct = generate_noisy_observations(
    times,
    clean_pops,
    trajectory_id="a_regular_noisy",
    model="A",
    noise_sigma=0.01,
    noise_seed=11,
    n_ref=100.0,
)

idx_tr, idx_va, idx_te = (
    obs_1pct["train_indices"],
    obs_1pct["val_indices"],
    obs_1pct["test_indices"],
)
print(f"Train points: {len(idx_tr)} (0 to {times[idx_tr[-1]]:.2f})")
print(f"Val points:   {len(idx_va)} ({times[idx_va[0]]:.2f} to {times[idx_va[-1]]:.2f})")
print(f"Test points:  {len(idx_te)} ({times[idx_te[0]]:.2f} to {times[idx_te[-1]]:.2f})")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axvspan(0, times[idx_tr[-1]], color="#e6e6ff", label="Train Split (60%)")
ax.axvspan(times[idx_tr[-1]], times[idx_va[-1]], color="#e6f2e6", label="Val Split (20%)")
ax.axvspan(times[idx_va[-1]], times[-1], color="#ffe6e6", label="Held-out Test (20%)")
ax.plot(times, clean_pops, "k-", lw=2, label="Clean Reference")
ax.scatter(
    times,
    obs_1pct["observed_population"],
    color="darkblue",
    s=20,
    label="Noisy Observations (1% noise)",
)
ax.set_title("Model A: Observation Splits and Noise Structure")
ax.set_xlabel("Time t")
ax.set_ylabel("Population N(t)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, ROOT / "artifacts/figures/wp05_demo_observations_1pct")
plt.show()

## 5.3 Baseline Inverse PINN on Clean Data (Ideal Parameter Recovery)

We first evaluate the Inverse PINN under ideal, noise-free conditions ($\sigma = 0.0$) to measure how closely the two-phase optimizer (Adam $\to$ L-BFGS) recovers the declared true values ($r=0.70, A=20.0, H=10.0$) from starting guesses ($r=1.0, A=30.0, H=5.0$). Numerical errors are reported rather than assuming exact recovery.

Physical parameters are constrained to valid ecological bounds via smooth bijective transformations:
$$r = \mathrm{softplus}(\tilde{r}) > 0, \quad A = K \cdot \sigma(\tilde{A}) \in (0, K), \quad H = \mathrm{softplus}(\tilde{H}) \ge 0$$
Training loss combines data fidelity on training observations ($t \le 4.8$) and the physics ODE residual evaluated at 300 collocation points. The network uses 2 hidden layers of 32 units with $\tanh$ activation and exact hard initial conditions $x(0) = x_0$.

**Live illustrations versus frozen benchmark.** These two single-case demos use parameter learning rate $5\times10^{-3}$, physics weight $0.5$ and at most 3000 L-BFGS iterations, with 500 Adam steps for clean data and up to 1200 for noisy data. Each fit has a 120-second CPU budget. The recorded benchmark in Section 5.5 retains its original shared configuration (parameter learning rate $10^{-3}$, physics weight $0.1$, 1000 Adam and 2500 L-BFGS steps, validation selection off); it is loaded, not retrained here.

In [ ]:
try:
    import jax  # noqa: F401  (availability check)
    import jax.numpy as jnp
    import matplotlib.pyplot as plt
    from IPython.display import display
    from jax.flatten_util import ravel_pytree

    import mm_allee.pinn.inverse as inv_mod
    from mm_allee.pinn.inverse import InversePINNTrainer, TrainableParameters
    from mm_allee.pinn.network import NetworkConfig, init_params

    known = {"known_initial_population": 45.0, "known_K": 100.0}
    ncfg = NetworkConfig(layers=(1, 32, 32, 1), activation="tanh", seed=1011)
    tp = TrainableParameters(
        "A",
        fit_params=["r", "A", "H"],
        known_conditions={"known_K": 100.0},
        initial_guesses={"r": 1.0, "A": 30.0, "H": 5.0},
    )

    # Track parameter trajectory across both Adam exploration and L-BFGS refinement
    dummy_net = init_params(ncfg)
    _, unravel_fn = ravel_pytree((dummy_net, tp.raw_params))
    lbfgs_history = {p: [] for p in tp.fit_params}

    orig_minimize = inv_mod.minimize

    def tracking_minimize(fun, x0, *args, **kwargs):
        user_cb = kwargs.get("callback")

        def wrapped_cb(intermediate_result):
            x_vec = getattr(intermediate_result, "x", intermediate_result)
            cand = unravel_fn(jnp.asarray(x_vec))
            phys = tp.get_physical_parameters(cand[1])
            for p in tp.fit_params:
                lbfgs_history[p].append(float(phys[p]))
            if user_cb is not None:
                # Preserve native logging and cancellation; do not swallow errors.
                user_cb(intermediate_result)

        kwargs["callback"] = wrapped_cb
        return orig_minimize(fun, x0, *args, **kwargs)

    trainer = InversePINNTrainer(
        network_config=ncfg,
        trainable_params=tp,
        lr_net=1e-3,
        lr_params=5e-3,
        lambda_data=1.0,
        lambda_phys=0.5,
        lbfgs_iters=3000,
        lbfgs_ftol=0.0,
        lbfgs_gtol=0.0,
        validation_selection=False,
        time_budget_s=120.0,
    )

    try:
        inv_mod.minimize = tracking_minimize
        pinn_clean = trainer.fit(
            obs_clean,
            known_conditions=known,
            epochs=500,
            patience=500,
            n_collocation=300,
            seed=1011,
        )
    finally:
        inv_mod.minimize = orig_minimize

    assert inv_mod.minimize is orig_minimize
    assert len(lbfgs_history["r"]) == pinn_clean.training_history["phase"].count("lbfgs")
    print(f"PINN Clean Fit Completed: {pinn_clean.status} in {pinn_clean.wall_seconds:.1f}s")
    print(
        "L-BFGS exit:", pinn_clean.optimizer["lbfgs_outcome"], pinn_clean.optimizer["lbfgs_message"]
    )
    print("Parameter Recovery on Clean Data:")
    for p in ["r", "A", "H"]:
        true_v = getattr(p_true, p)
        est_v = pinn_clean.estimated_parameters[p]
        rel_err = abs(est_v - true_v) / true_v * 100
        print(f"  {p}: True = {true_v:.2f}, PINN Est = {est_v:.4f} (Error: {rel_err:.2f}%)")

    # Combine Adam and L-BFGS history so curves end exactly at final values
    adam_len = len(pinn_clean.training_history["param_r"])
    full_history = {}
    for p in tp.fit_params:
        adam_vals = list(pinn_clean.training_history[f"param_{p}"])
        lbfgs_vals = list(lbfgs_history[p])
        combined = adam_vals + lbfgs_vals
        if not combined:
            raise RuntimeError("No parameter trajectory was recorded")
        full_history[p] = combined

    total_steps = list(range(1, len(full_history["r"]) + 1))
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4.2))

    # Parameter r
    ax1.plot(total_steps, full_history["r"], color="#1f77b4", lw=2, label="Estimated r")
    ax1.axhline(p_true.r, color="black", linestyle="--", lw=1.5, label=f"True r = {p_true.r:.2f}")
    if adam_len < len(total_steps):
        ax1.axvline(adam_len, color="#b3b3b3", linestyle=":", label="L-BFGS Phase")
    ax1.scatter(
        [total_steps[-1]],
        [pinn_clean.estimated_parameters["r"]],
        color="red",
        s=50,
        zorder=5,
        label=f"Final = {pinn_clean.estimated_parameters['r']:.4f}",
    )
    ax1.set_title("Growth Rate (r) Convergence")
    ax1.set_xlabel("Optimization Step (Adam + L-BFGS)")
    ax1.set_ylabel("Parameter Value")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Parameter A
    ax2.plot(total_steps, full_history["A"], color="#2ca02c", lw=2, label="Estimated A")
    ax2.axhline(p_true.A, color="black", linestyle="--", lw=1.5, label=f"True A = {p_true.A:.2f}")
    if adam_len < len(total_steps):
        ax2.axvline(adam_len, color="#b3b3b3", linestyle=":", label="L-BFGS Phase")
    ax2.scatter(
        [total_steps[-1]],
        [pinn_clean.estimated_parameters["A"]],
        color="red",
        s=50,
        zorder=5,
        label=f"Final = {pinn_clean.estimated_parameters['A']:.4f}",
    )
    ax2.set_title("Allee Threshold (A) Convergence")
    ax2.set_xlabel("Optimization Step (Adam + L-BFGS)")
    ax2.set_ylabel("Parameter Value")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # Parameter H
    ax3.plot(total_steps, full_history["H"], color="#ff7f0e", lw=2, label="Estimated H")
    ax3.axhline(p_true.H, color="black", linestyle="--", lw=1.5, label=f"True H = {p_true.H:.2f}")
    if adam_len < len(total_steps):
        ax3.axvline(adam_len, color="#b3b3b3", linestyle=":", label="L-BFGS Phase")
    ax3.scatter(
        [total_steps[-1]],
        [pinn_clean.estimated_parameters["H"]],
        color="red",
        s=50,
        zorder=5,
        label=f"Final = {pinn_clean.estimated_parameters['H']:.4f}",
    )
    ax3.set_title("Harvest Quota (H) Convergence")
    ax3.set_xlabel("Optimization Step (Adam + L-BFGS)")
    ax3.set_ylabel("Parameter Value")
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    plt.suptitle("Clean Data: Recorded Parameter Trajectory (Model A)", fontsize=13, y=1.02)
    plt.tight_layout()
    save_figure(fig, ROOT / "artifacts/figures/wp05_demo_clean_parameter_trajectory")
    with (wp05_demo_dir / "clean_parameter_trajectory.csv").open("w", newline="") as handle:
        writer = csv.writer(handle, lineterminator="\n")
        writer.writerow(["step", "phase", "r", "A", "H"])
        for i, step in enumerate(total_steps):
            writer.writerow(
                [
                    step,
                    "adam" if i < adam_len else "lbfgs",
                    *[full_history[p][i] for p in ["r", "A", "H"]],
                ]
            )
    display(fig)
    plt.close(fig)
except ImportError:
    print(
        "JAX is not installed in this environment; skipping PINN training. Full code is implemented in mm_allee.pinn."
    )

## 5.4 Parameter Estimation Under Measurement Noise: ODE Least Squares vs PINN

We now evaluate both methods on **1% measurement noise** ($\sigma = 0.01$, noise standard deviation $= 1.0$) starting from the identical initial guesses ($r=1.0, A=30.0, H=5.0$):
- **Conventional ODE Least Squares (`ODE_LS`):** Integrates the ODE with adaptive RK45 and optimizes parameters using SciPy's Trust Region Reflective algorithm on the training split ($t \le 4.8$).
- **Inverse PINN:** Jointly optimizes network weights and physical parameters using the two-phase Adam + L-BFGS optimizer on the training split and collocation grid.

Both methods run on the exact same noisy observations to provide a direct, fair comparison.

In [ ]:
# 1. Conventional ODE Least-Squares Fit
ode_res = fit_ode_least_squares(
    "A",
    observations=obs_1pct,
    known_conditions={"known_K": 100.0, "known_initial_population": 45.0},
    fit_params=["r", "A", "H"],
    initial_guess={"r": 1.0, "A": 30.0, "H": 5.0},
    n_ref=100.0,
)

# 2. Inverse PINN Fit
pinn_noisy_res = None
try:
    import jax  # noqa: F401

    from mm_allee.pinn.inverse import InversePINNTrainer, TrainableParameters
    from mm_allee.pinn.network import NetworkConfig

    ncfg = NetworkConfig(layers=(1, 32, 32, 1), activation="tanh", seed=1011)
    tp = TrainableParameters(
        "A",
        fit_params=["r", "A", "H"],
        known_conditions={"known_K": 100.0},
        initial_guesses={"r": 1.0, "A": 30.0, "H": 5.0},
    )
    trainer_noisy = InversePINNTrainer(
        network_config=ncfg,
        trainable_params=tp,
        lr_net=1e-3,
        lr_params=5e-3,
        lambda_data=1.0,
        lambda_phys=0.5,
        lbfgs_iters=3000,
        lbfgs_ftol=0.0,
        lbfgs_gtol=0.0,
        validation_selection=True,  # Rohan's noisy-demo checkpoint policy; benchmark remains False.
        time_budget_s=120.0,
    )
    pinn_noisy_res = trainer_noisy.fit(
        obs_1pct,
        known_conditions={"known_initial_population": 45.0, "known_K": 100.0},
        epochs=1200,
        n_collocation=300,
        seed=1011,
    )
except ImportError:
    pass

# 3. Comparative Summary Table
print("=" * 86)
print(f"{'Model A Parameter Recovery Under 1% Measurement Noise (sigma = 0.01)':^86s}")
print("=" * 86)
print(
    f"{'Parameter':15s}{'True Value':>12s}{'ODE_LS Estimate (Error)':>30s}{'PINN Estimate (Error)':>28s}"
)
print("-" * 86)

for p in ["r", "A", "H"]:
    tv = getattr(p_true, p)
    ode_v = ode_res.estimated_parameters[p]
    ode_err = abs(ode_v - tv) / tv * 100
    ode_str = f"{ode_v:.4f} ({ode_err:.2f}%)"
    if pinn_noisy_res is not None:
        pinn_v = pinn_noisy_res.estimated_parameters[p]
        pinn_err = abs(pinn_v - tv) / tv * 100
        pinn_str = f"{pinn_v:.4f} ({pinn_err:.2f}%)"
    else:
        pinn_str = "N/A (JAX required)"
    print(f"{p:15s}{tv:>12.2f}{ode_str:>30s}{pinn_str:>28s}")

print("-" * 86)
ode_time_str = f"{ode_res.wall_seconds:.3f}s"
pinn_time_str = f"{pinn_noisy_res.wall_seconds:.2f}s" if pinn_noisy_res is not None else "N/A"
print(f"{'Runtime':15s}{'-':>12s}{ode_time_str:>30s}{pinn_time_str:>28s}")

ode_rmse_str = f"{ode_res.test_metrics['rmse']:.4f}"
pinn_rmse_str = (
    f"{pinn_noisy_res.test_metrics['rmse']:.4f}" if pinn_noisy_res is not None else "N/A"
)
print(f"{'Test RMSE':15s}{'-':>12s}{ode_rmse_str:>30s}{pinn_rmse_str:>28s}")
print("=" * 86)

print("Noisy-demo validation selection:", trainer_noisy.validation_selection)
print(
    "L-BFGS exit:",
    pinn_noisy_res.optimizer["lbfgs_outcome"],
    pinn_noisy_res.optimizer["lbfgs_message"],
)
demo_record = {
    "scope": "Two illustrative Model A regular fits; not a rerun of the frozen benchmark",
    "source": "Rohan's supplied WP05 notebook, integrated without WP01-WP04 changes",
    "known_conditions": {"known_K": 100.0, "known_initial_population": 45.0},
    "true_parameters": {p: getattr(p_true, p) for p in ["r", "A", "H"]},
    "initial_guesses": {"r": 1.0, "A": 30.0, "H": 5.0},
    "noise_seed": 11,
    "fit_seed": 1011,
    "n_ref": 100.0,
    "demo_settings": {
        "lr_net": 1e-3,
        "lr_params": 5e-3,
        "lambda_phys": 0.5,
        "n_collocation": 300,
        "lbfgs_iters": 3000,
        "time_budget_s": 120.0,
    },
    "clean": {
        "noise_sigma": 0.0,
        "adam_epochs_requested": 500,
        "validation_selection": False,
        "estimates": pinn_clean.estimated_parameters,
        "optimizer": pinn_clean.optimizer,
        "status": pinn_clean.status,
        "success": pinn_clean.success,
        "wall_seconds": pinn_clean.wall_seconds,
    },
    "noisy": {
        "noise_sigma": 0.01,
        "adam_epochs_requested": 1200,
        "validation_selection": True,
        "estimates": pinn_noisy_res.estimated_parameters,
        "optimizer": pinn_noisy_res.optimizer,
        "status": pinn_noisy_res.status,
        "success": pinn_noisy_res.success,
        "test_rmse": pinn_noisy_res.test_metrics["rmse"],
        "wall_seconds": pinn_noisy_res.wall_seconds,
    },
    "ode_noisy": {
        "estimates": ode_res.estimated_parameters,
        "status": ode_res.status,
        "test_rmse": ode_res.test_metrics["rmse"],
        "wall_seconds": ode_res.wall_seconds,
    },
    "trajectory_rows": len(total_steps),
    "adam_trace_rows": adam_len,
    "lbfgs_trace_rows": len(lbfgs_history["r"]),
}
(wp05_demo_dir / "summary.json").write_text(json.dumps(demo_record, indent=2) + "\n")
print("WP05 demo artifacts:", wp05_demo_dir)

## 5.5 Final Sweep Benchmark (108 Recorded Runs)

The full benchmark experiment evaluates **108 total fits** (54 PINN + 54 ODE least squares) spanning:
1. **6 Dynamical Regimes:**
   - **Model A:** `a_regular` (strong-Allee persistence), `a_near_fold` (near saddle-node tipping point), `a_extinction` (collapse below Allee threshold).
   - **Model B:** `b_regular` (proportional harvest persistence), `b_near_fold` (near threshold), `b_decline` (steep transient decline).
2. **3 Noise Levels:** dimensionless $\sigma \in \{0.0, 0.01, 0.05\}$ with noise standard deviation $\sigma N_{\text{ref}}$ (clean, 1%, and 5% additive Gaussian noise).
3. **3 Random Seeds:** Seeds `{11, 22, 33}` per condition to verify stochastic reproducibility.

The table below summarizes the **clean-data baseline ($\sigma = 0$)** across the 3 seeds for each regime:
- **Parameter Recovered (PINN):** Counts how many of the 3 random seeds successfully recovered **every** unknown parameter within 5% relative error ($\le 0.05$).
- **PINN / ODE_LS Median Error:** The median across all 3 seeds of the worst-case parameter relative error for that case.

**Optimizer reporting.** The recorded benchmark has 54 PINN iteration-limit exits. Its legacy `converged` label denotes training completion, not reaching an L-BFGS tolerance. The clean/noisy live demos above use different budgets and report their own optimizer exits.

In [ ]:
import csv  # noqa: F811  (an earlier fragment imports csv in the assembled notebook)
import statistics

rows = list(csv.DictReader(open("artifacts/tables/wp05_parameter_metrics.csv")))
fits = {}
for r in rows:
    if float(r["noise_sigma"]) == 0.0:
        fits.setdefault((r["method"], r["run_id"]), {})[r["parameter"]] = float(r["relative_error"])

cases = ["a_regular", "b_regular", "a_extinction", "b_near_fold", "a_near_fold", "b_decline"]
print(
    f"{'Dynamical Case':16s}{'Parameter Recovered (PINN)':>28s}{'PINN Median Err':>18s}{'ODE_LS Median Err':>20s}"
)
print("-" * 84)

total = 0
for c in cases:
    pinn = [max(v.values()) for (m, rid), v in fits.items() if m == "PINN" and f"wp05_{c}_" in rid]
    ode = [max(v.values()) for (m, rid), v in fits.items() if m == "ODE_LS" and f"wp05_{c}_" in rid]
    ok = sum(e <= 0.05 for e in pinn)
    total += ok
    print(
        f"{c:16s}{f'{ok}/3 seeds':>28s}{statistics.median(pinn):>18.1e}{statistics.median(ode):>20.1e}"
    )

print("-" * 84)
print(f"Total PINN cases fully recovered (all parameters <= 5% error): {total}/18 clean fits")

![Parameter recovery versus noise](../artifacts/figures/wp05_recovery_noise.png)

![Held-out windows, Model A regular](../artifacts/figures/wp05_heldout_failure.png)

The first figure averages over all six cases and seeds, so its error bars are dominated by the badly conditioned cases. The second shows Model A regular (top row noise-free, bottom row noisy); the regular trajectories are at equilibrium by $t=4.8$, so the held-out window is an undemanding physics-supported prediction test: collocation covers the full $[0,8]$ interval, including the held-out window, so this is not extrapolation outside the physics domain.

### 5.6 Diagnostic Analysis: Why 8 PINN Fits Failed and Why ODE_LS Outperforms PINNs

Across the 18 clean-data fits ($\sigma = 0$), **`ODE_LS` recovers 18/18 within the declared 5% criterion**, whereas **PINN recovers 10/18** ($a\_regular$: 3/3, $b\_regular$: 3/3, $b\_near\_fold$: 3/3, $a\_extinction$: 1/3, $a\_near\_fold$: 0/3, $b\_decline$: 0/3).

This performance divergence highlights measured differences between classical numerical ODE solvers and neural physics-informed optimization:

---

#### 1. Why Eight PINN Fits Did Not Meet the Parameter-Recovery Criterion While the Other 10 Succeeded?

The 18 benchmark runs give the following recovery counts; conditioning alone does not decide success:

* **The 10 Successful Fits ($a\_regular$ 3/3, $b\_regular$ 3/3, $b\_near\_fold$ 3/3, $a\_extinction$ 1/3):**
  * **$a\_regular$ and $b\_regular$ (6/6 recovered):** Trajectories exhibit smooth, monotonic sigmoidal growth with low Jacobian condition numbers ($\kappa(J) \in [62, 340]$). The physical parameters $(r, A, H/h)$ exert distinct, decoupled influences across the transient phase, allowing data and physics loss gradients to remain balanced.
  * **$b\_near\_fold$ (3/3 recovered despite large condition number $\kappa \approx 10^3$):** A large condition number indicates that the loss surface is *ill-conditioned* (an elongated, narrow valley), **not mathematically unidentifiable**. In Model B, proportional harvesting ($hN$) scales with population and factors out $N$. The trajectory retains dynamic curvature over time. Limited-memory quasi-Newton refinement can help navigate poorly conditioned directions, but this study did not measure an exact Hessian or isolate the cause of that success.

* **The 8 Unrecovered Fits ($a\_near\_fold$ 3 failures, $b\_decline$ 3 failures, $a\_extinction$ 2 failures):**
  * **$a\_near\_fold$ (0/3 recovered):** Fixed quota harvesting near the saddle-node fold creates a severe dynamic bottleneck where $dN/dt \approx 0$ for an extended duration ("ghost of the fold"). Trajectory sensitivity collapses: $r$ and $A$ become strongly collinear ($r/A \approx \text{const}$). The recorded fits have small trajectory errors and large parameter errors. Flat dynamics and correlated sensitivities are plausible explanations; gradient magnitudes were not measured in this experiment.
  * **$b\_decline$ (0/3 recovered) and $a\_extinction$ (2/3 failed):** Both cases feature steep, rapid transient collapses followed by a long, flat tail near zero population. Limited transient information and collocation resolution are possible contributors, but their causal effect on these inverse failures has not been isolated here.

---

#### 2. Why Does `ODE_LS` Outperform PINNs Across All Cases?

| Metric / Property | Classical `ODE_LS` | Physics-Informed Neural Network (PINN) |
| :--- | :--- | :--- |
| **Search Space** | Strictly **3 dimensions** ($\theta = [r, A, H] \in \mathbb{R}^3$) | **1,000+ dimensions** ($W, b \in \mathbb{R}^{1000}$ plus $\theta \in \mathbb{R}^3$) |
| **Physics Enforcement** | **Numerical ODE constraint:** Adaptive RK45 with declared solver tolerances; solver error is not zero or machine precision. | **Soft penalty:** Approximated as a secondary loss term $\lambda_{\text{phys}} \mathcal{L}_{\text{phys}}$ evaluated at discrete collocation points. |
| **Loss Landscape** | Bounded nonlinear least squares using Trust Region Reflective. These results do not establish convexity or a unique global optimisation basin. | Multi-objective composite landscape with saddle points, flat plateaus, and spurious local minima. |
| **Gradient Dynamics** | The implementation estimates the residual Jacobian numerically; analytical sensitivities are not supplied to this optimiser. | **Gradient Stiffness / Spectral Bias:** Network weights $W$ fit observation points ($\mathcal{L}_{\text{data}} \to 0$) far faster than physical parameters $\theta$ can minimize the ODE residual ($\mathcal{L}_{\text{phys}}$), trapping $\theta$ in suboptimal values. |

> **Key Scientific Takeaway:** Parameter Error $\ne$ Trajectory Error. A PINN can fit observation points to $10^{-4}$ trajectory residual while parameter estimates remain 30% to 100% in error. `ODE_LS` avoids this failure mode because it directly evaluates the forward ODE vector field rather than an unconstrained neural surrogate.

---

#### 3. How Can the 8 Failing PINN Fits Be Recovered?

The following are candidate follow-up methods, not demonstrated fixes for these eight runs. They require new controlled experiments:
1. **Adaptive Loss Weighting (SoftAdapt / GradNorm):** Instead of static weights ($\lambda_{\text{data}}=1.0, \lambda_{\text{phys}}=0.5$), dynamically balance the gradients $\nabla_W \mathcal{L}_{\text{data}}$ and $\nabla_\theta \mathcal{L}_{\text{phys}}$ at each epoch so neither objective dominates.
2. **Residual-based Adaptive Collocation (RAR):** Dynamically concentrate collocation points in the steep collapse and inflection regions rather than distributing them uniformly across time.
3. **Curriculum Training / Time-Domain Decomposition:** Train the network sequentially over advancing time windows ($[0, t_1] \to [0, t_2] \to [0, t_{\text{end}}]$) to resolve initial transient dynamics before optimizing on flat asymptotic equilibria.

**Scope and sources.** Exact N0 is independently known in these experiments, including noisy runs. The ODE observation model uses declared zero padding after candidate extinction. Mechanistic explanations above are hypotheses where not directly tested. See [SciPy L-BFGS-B](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-lbfgsb.html) for the limited-memory approximation and [SciPy least_squares](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.least_squares.html) for the implemented TRF method.

## 5.7 Key Scientific Takeaways

1. **Parameter Error $\ne$ Trajectory Error:** A PINN can fit observations to $10^{-4}$ trajectory residual while parameter estimates remain substantially inaccurate in ill-conditioned regimes.
2. **Classical ODE Baselines are Strong:** For low-dimensional scalar ODEs, SciPy nonlinear least squares is faster, more robust, and recovers parameters across all six regimes on clean data.
3. **Structural Identifiability Precedes Optimization:** Model B with all four parameters $(r,A,K,h)$ unknown is structurally ambiguous; no optimization algorithm can break this ambiguity without independent constraints (such as known carrying capacity $K$).
4. **Noise Floor as a Convergence Metric:** Comparing training RMSE to the noise standard deviation ($\text{RMSE} / \sigma N_{\text{ref}} \approx 0.9$) measures fit relative to the noise scale; it does not by itself establish parameter accuracy or extraction of all available information.